In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun

import os

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    google_api_key=GOOGLE_API_KEY
)

llm_c = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.9,
    google_api_key=GOOGLE_API_KEY
)

llm_p = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    google_api_key=GOOGLE_API_KEY
)

search = DuckDuckGoSearchRun()

In [14]:
tools = [search]

research_agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
)

# agent.run("cricket")
# topic = input("Enter a topic to research: ")

# — Planning-specific tools

def break_into_steps(task: str) -> str:
    """Breaks a goal into ordered steps."""
    return f"Breaking down: {task} into sequential steps for planning."

def estimate_timeline(task: str) -> str:
    """Estimates time required for a given task or plan."""
    return f"Estimating timeline for: {task}"

planning_tools = [
    Tool(
        name="TaskBreaker",
        func=break_into_steps,
        description=(
            "Use this to decompose a complex goal into smaller, "
            "manageable steps. Input should be a goal or objective."
        ),
    ),
    Tool(
        name="TimelineEstimator",
        func=estimate_timeline,
        description=(
            "Use this to estimate how long a task or full plan will take."
            "Input should be as task description."
        ),
    ),
]
# Planning Agent

planning_agent = initialize_agent(
    planning_tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
)

In [15]:
# writing-special-tools

def draft_content(topic: str) -> str:
    """Drafts an initial version of content on a given topic."""
    return f"Drafting initial content for: {topic}"

def generate_title(topic: str) -> str:
    """Generates compelling titles or headlines for the content."""
    return f"Generating title options for: {topic}"

writing_tools = [
    search,
    Tool(
        name="DraftWriter",
        func=draft_content,
        description="""
        Use this to create a first draft on web topics.
        Input should be the topic or title of the content.
        """
    ),
    Tool(
        name="TitleGenerator",
        func=generate_title,
        description="""
        Use this to generate main catchy titles or headlines.
        Input should be one topic or summary of the content.
        """
    ),
]

# — Writing Agent
writing_agent = initialize_agent(
    writing_tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
)

In [ ]:
# — Editing-specific tools
def fix_grammar(text: str) -> str:
    """Fixes grammar, punctuation, and spelling errors."""
    return f"Fixing grammar and punctuation in: '{text[:100]}'..."

def plagiarism_check(text: str) -> str:
    """Flags potentially plagiarized or unoriginal sections."""
    return f"Scanning for plagiarism in: {text[:100]}..."

editing_tools = [
    search,

    Tool(
        name="GrammarFixer",
        func=fix_grammar,
        description=(
            "Use this to correct grammar, punctuation, and spelling. "
            "Input should be the text to fix."
        ),
    ),

    Tool(
        name="PlagiarismChecker",
        func=plagiarism_check,
        description=(
            "Use this to detect potentially plagiarized content. "
            "Input should be the text to scan."
        ),
    ),
]
# Editing Agent
editing_agent = initialize_agent(
    editing_tools,
    llm,
    agent= AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
)

# Run
#article_to_edit = input("Paste your article/content to edit: ")

In [17]:
# Outline specific tools

def extract_key_sections(report: str) -> str:
    """Extracts all sections and headings from the report."""
    return f"Extracting key sections from report: {report[:100]}..."

def map_to_slides(sections: str) -> str:
    """Maps each report section to a corresponding slide."""
    return f"Mapping sections to slides: {sections[:100]}..."

def condense_to_bullets(section: str) -> str:
    """Condenses each section into 3-5 slide-ready bullet points."""
    return f"Condensing to bullet points: {section[:100]}..."

def write_slide_title(section: str) -> str:
    """Writes a smart, punchy title for each slide."""
    return f"Writing slide title for: {section[:100]}..."

def write_transition_notes(slide: str) -> str:
    """Adds transition phrases between slides for smooth flow."""
    return f"Writing transition notes for: {slide[:100]}..."

def identify_data_slides(report: str) -> str:
    """Identifies sections that should become charts or data visuals."""
    return f"Identifying data/chart opportunities in: {report[:50]}..."

def create_title_slide(topic: str) -> str:
    """Creates the opening title slide content."""
    return f"Creating title slide for: {topic}"

def create_summary_slide(report: str) -> str:
    """Creates a final summary/key takeaway slide."""
    return f"Creating summary slide from: {report[:100]}..."

def number_slides(outline: str) -> str:
    """Numbers and sequences all slides in correct order."""
    return f"Numbering and sequencing slides: {outline[:100]}..."

def estimate_slide_count(report: str) -> str:
    """Estimates how many slides the report will need."""
    return f"Estimating slide count for report: {report[:100]}..."
outline_tools = [
    search,
    Tool(
        name="SlideMapper",
        func=map_to_slides,
        description="""
        Use this to map each report section to a slide.
        Input should be the list of extracted sections.
        """,
    ),
    Tool(
        name="SlideTitleWriter",
        func=write_slide_title,
        description="""
        Use this to write a short punchy title for each slide.
        Input should be the section heading or topic of the slide.
        """,
    ),
    Tool(
        name="SlideCountEstimator",
        func=estimate_slide_count,
        description="""
        Use this to estimate the total number of slides needed.
        Input should be the full report text.
        """,
    ),
]

# — Presentation Outline Agent —

presentation_outline_agent = initialize_agent(
    outline_tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
)

# — Run —
# final_report = input("Paste your final edited report: ")


In [18]:
# FULL 5-AGENT PIPELINE

# = Single user Input

goal = input("Enter your goal: ")

# AGENT 1 - RESEARCH AGENT
# Input : user goal
# Output : research_output

prompt = f"""
USER GOAL:
{goal}

Research ONLY this goal.
Gather recent information, facts, statistics, and best practices.
Provide a detailed summary.
"""

research_output = research_agent.run(prompt)
print("** AGENT 1 - Research Done\n")

# AGENT 2 - PLANNING AGENT
# Input : user goal
# Output : planning_output

planning_prompt = f"""
USER GOAL:
{goal}

RESEARCH SUMMARY:
{research_output}

Create a detailed actionable plan.

Requirements:
- Break into phases
- Use TaskBreaker
- Estimate timeline
- Provide deadlines
"""

planning_output = planning_agent.run(planning_prompt)
print("** AGENT 2 - Planning Done\n")

# AGENT 3 - WRITING AGENT
# Input : research_output + planning_output
# Output : writing_output

writing_prompt = f"""
ORIGINAL GOAL:
{goal}

RESEARCH SUMMARY:
{research_output}

PLAN:
{planning_output}

IMPORTANT:
- Do not change the topic.
- Do not introduce a new subject.
- Use the research and plan provided.

TASK:
Write a detailed article.

Output:
- Title
- Introduction
- Main Sections
- Conclusion
"""

writing_output = writing_agent.run(writing_prompt)

print("** AGENT 3 - Writing Done\n")

# AGENT 4 - EDITING AGENT
# Input = writing_output
# Output = editing_output

editing_prompt = f"""
ORIGINAL GOAL:
{goal}

PLAN:
{planning_output}

ARTICLE DRAFT:
{writing_output}

IMPORTANT:
The article must remain aligned with the original goal and plan.

TASK:
Edit and polish the article.

Requirements:
- Correct grammar and spelling.
- Improve readability.
- Improve transitions between sections.
- Remove redundancy.
- Maintain all key facts.
- Preserve the original meaning.
- Do not introduce unrelated topics.
- Do not shorten the article significantly.

Use GrammarFixer.
Use PlagiarismChecker.

Output only the final edited article.
"""

editing_output = editing_agent.run(editing_prompt)

print("** AGENT 4 - Editing Done\n")

# AGENT 5 - PRESENTATION OUTLINE AGENT
# Input = editing_output
# Output = final_presentation

outline_prompt = f"""
ORIGINAL GOAL:
{goal}

PLAN:
{planning_output}

FINAL REPORT:
{editing_output}

TASK:
Convert the report into a presentation-ready slide deck.

IMPORTANT:
- Stay aligned with the original goal.
- Do not introduce new topics.
- Use the report as the primary source.
- Use the plan only for structure if needed.

Follow these steps:
1. Determine the number of slides needed.
2. Map content to slides logically.
3. Create a concise title for each slide.

Output format:

SLIDE [N] – [TITLE]

- [Point]&#58; [1-2 sentence explanation]
- [Point]&#58; [1-2 sentence explanation]
- [Point]&#58; [1-2 sentence explanation]

Rules:
- Exactly 3 bullets per slide.
- Bullet labels should be concise (3-6 words).
- Explanations should provide context, evidence, or examples.
- No commentary outside the slide format.
"""

#presentation_outline_age   nt.run(outline_prompt)

final_presentation = presentation_outline_agent.run(outline_prompt)
print("** AGENT 5 – Presentation Done\n")

print("-" * 60)
print(" FINAL PRESENTATION OUTLINE")
print("-" * 60)
print(final_presentation)




> Entering new AgentExecutor chain...
Thought: I will search for a guide on creating a chat app in C++ using networking concepts.
Action: duckduckgo_search
Action Input: create chat app in c++ using networkingO
Observation: Feb 26, 2025 · Learn to build a real-time chat application using C ++ and UDP. This tutorial covers network programming, socket creation, and message handling for efficient communication. Learn how to create a simple chat application using C ++ and TCP/IP networking . A beginner-friendly TCP chat application built in C ++17 with multithreaded server and client support. This project demonstrates basic network programming concepts including socket programming, threading, and client-server communication. This is a simple real-time chat application implemented in C ++ using WinSock for network communication. It includes both server and client components, enabling multiple clients to connect to the server and communicate with each other. This project demonstrates how t